In [7]:
import itertools
import shutil
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import polars.selectors as cs
import seaborn as sns
from tqdm import tqdm

from polars_ml import Pipeline

out_dir = Path("out")
out_dir.mkdir(exist_ok=True, parents=True)

pl.Config.set_tbl_rows(100)

train_df = pl.read_csv("../../data/raw/train.csv")
test_df = pl.read_csv("../../data/raw/test.csv")
submit_df = pl.read_csv("../../data/raw/sample_submission.csv")

train_df = train_df.drop("id").cast({"date": pl.Date})
test_df = test_df.drop("id").cast({"date": pl.Date})

gdp_df = pl.read_csv("../../data/external/gdp.csv")

train_df.head()

date,country,store,product,num_sold
date,str,str,str,f64
2010-01-01,"""Canada""","""Discount Stickers""","""Holographic Goose""",null
2010-01-01,"""Canada""","""Discount Stickers""","""Kaggle""",973.0
2010-01-01,"""Canada""","""Discount Stickers""","""Kaggle Tiers""",906.0
2010-01-01,"""Canada""","""Discount Stickers""","""Kerneler""",423.0
2010-01-01,"""Canada""","""Discount Stickers""","""Kerneler Dark Mode""",491.0


In [8]:
countries = train_df["country"].unique().to_list()
stores = train_df["store"].unique().to_list()
products = train_df["product"].unique().to_list()

train_df = train_df.with_columns(
    pl.col("date").dt.year().alias("year"),
    pl.col("date").dt.month().alias("month"),
    pl.col("date").dt.week().alias("week"),
    pl.col("date").dt.weekday().alias("weekday"),
    pl.col("date").dt.day().alias("day"),
).join(
    gdp_df.unpivot(countries, index="year", variable_name="country", value_name="gdp"),
    on=["year", "country"],
)

In [9]:
pp = Pipeline().standard_scale(
    "num_sold", by="country", component_name="num_sold_scale_by_country"
)
df = pp.fit_transform(train_df)

countries = train_df["country"].unique().to_list()
stores = train_df["store"].unique().to_list()
products = train_df["product"].unique().to_list()

out_dir = Path("./out/num_sold_density")
shutil.rmtree(out_dir, ignore_errors=True)
out_dir.mkdir(exist_ok=True, parents=True)
for category in ["country", "store", "product"]:
    tmp = df.drop_nulls().group_by("date", category).agg(pl.col("num_sold").mean())
    fig, ax = plt.subplots(figsize=(12, 8))
    sns.histplot(tmp, x="num_sold", hue=category, kde=True, alpha=0.2, ax=ax)
    ax.set_title(f"num_sold density by {category}")
    ax.set_xlabel("num_sold")
    ax.set_ylabel("density")
    fig.savefig(out_dir / f"{category}.png")
    fig.clear()
    plt.close(fig)


In [10]:
out_dir = Path("./out/num_sold_over_date")
shutil.rmtree(out_dir, ignore_errors=True)
out_dir.mkdir(exist_ok=True, parents=True)
for category in ["country", "store", "product"]:
    tmp = df.drop_nulls().group_by("date", category).agg(pl.col("num_sold").mean())
    fig, ax = plt.subplots(figsize=(20, 8))
    sns.lineplot(data=tmp, x="date", y="num_sold", hue=category, ax=ax)
    ax.set_title(f"num_sold over date by {category}")
    ax.set_xlabel("date")
    ax.set_ylabel("num_sold")
    fig.savefig(out_dir / f"{category}.png")
    fig.clear()
    plt.close(fig)

In [11]:
out_dir = Path("./out/gdp_vs_num_sold")
shutil.rmtree(out_dir, ignore_errors=True)
out_dir.mkdir(exist_ok=True, parents=True)
for category in ["country", "store", "product"]:
    tmp = (
        df.drop_nulls()
        .group_by("year", "country", "store", "product")
        .agg(pl.col("num_sold").mean(), pl.col("gdp").mean())
    )
    fig, ax = plt.subplots(figsize=(10, 10))
    sns.scatterplot(
        tmp, x="gdp", y="num_sold", hue=category, edgecolor=None, s=5, ax=ax
    )
    ax.set_title(f"{category} vs num_sold vs GDP")
    ax.set_xlabel("GDP")
    ax.set_ylabel("num_sold")
    plt.savefig(out_dir / f"{category}.png")
    fig.clear()
    plt.close(fig)


In [16]:
from datetime import datetime

from sklearn.linear_model import LinearRegression

train_adj_df = pl.read_parquet("../../data/processed/train_num_sold_adj.parquet")
pp = Pipeline().standard_scale(
    "num_sold_adj", by="country", component_name="num_sold_adj_scale_by_country"
)
train_adj_df = pp.fit_transform(train_adj_df)

start_date = datetime(2010, 1, 1)

train_adj_df = train_adj_df.with_columns(
    (pl.col("date") - start_date).dt.total_days().alias("ordinal_day")
).with_columns(
    pl.col("ordinal_day").mul(2 * np.pi / 365).sin().alias("sin_period_year"),
    pl.col("ordinal_day").mul(2 * np.pi / 365).cos().alias("cos_period_year"),
    pl.col("ordinal_day").mul(2 * np.pi / 365 / 2).sin().alias("sin_period_2_years"),
    pl.col("ordinal_day").mul(2 * np.pi / 365 / 2).cos().alias("cos_period_2_years"),
    ((pl.col("day") - 1) // 7 + 1).alias("week_of_month"),
)
tmp = (
    train_adj_df.drop(
        "date",
        "num_sold",
        "year",
        "month",
        "week",
        # "weekday",
        "day",
        "week_of_month",
        "gdp",
        "coef",
        "gdp_linear_pred",
        "ordinal_day",
    )
    .to_dummies(["country", "store", "product", "weekday"])
    .with_columns(
        (pl.col("num_sold_adj") - pl.col("num_sold_adj").mean())
        / pl.col("num_sold_adj").std()
    )
)
tmp = Pipeline().polynomial(cs.exclude("num_sold_adj"), degree=2).transform(tmp)  # type: ignore
X = tmp.drop_nulls().drop("num_sold_adj").to_numpy()
y = tmp.drop_nulls()["num_sold_adj"].to_numpy()

model = LinearRegression()
model.fit(X, y)


LinearRegression()

In [17]:
X = tmp.drop("num_sold_adj").to_numpy()
y_pred = model.predict(X)

df = train_adj_df.with_columns(
    pl.Series("num_sold_adj_pred", y_pred) * pl.col("num_sold_adj").std()
    + pl.col("num_sold_adj").mean()
)

countries = train_df["country"].unique().to_list()
stores = train_df["store"].unique().to_list()
products = train_df["product"].unique().to_list()
categories = {
    "country": countries,
    "store": stores,
    "product": products,
}

out_dir = Path("./out/num_sold_pred_over_date")
shutil.rmtree(out_dir, ignore_errors=True)
out_dir.mkdir(exist_ok=True, parents=True)
for category in ["country", "store", "product"]:
    tmp2 = (
        df.drop_nulls()
        .group_by("date", category)
        .agg(pl.col("num_sold_adj").mean(), pl.col("num_sold_adj_pred").mean())
    )
    for cat in categories[category]:
        tmp3 = tmp2.filter(pl.col(category) == cat)
        fig, ax = plt.subplots(figsize=(50, 8))
        sns.lineplot(data=tmp3, x="date", y="num_sold_adj", ax=ax)
        sns.lineplot(data=tmp3, x="date", y="num_sold_adj_pred", ax=ax)
        ax.set_title(f"num_sold_adj over date by {category}={cat}")
        ax.set_xlabel("date")
        ax.set_ylabel("num_sold_adj")
        fig.savefig(out_dir / f"{category}_{cat}.png")
        fig.clear()
        plt.close(fig)
